In [4]:
import requests
import pandas as pd
import time
import os
from datetime import datetime


In [5]:
INPUT_CSV = "/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv"

In [6]:
df_aqi = pd.read_csv(INPUT_CSV)
df_aqi.head()

,Source,ID,Timestamp,Name,Latitude,Longitude,AQI,PM2.5,PM10,CO,NO2,O3,SO2,Temperature,Humidity,Pressure,Wind Speed
0,gov,28560877461938780203765592307,08/04/2025 14:00,Hà Nội: 556 Nguyễn Văn Cừ (KK),21.0491,105.8831,134,134.333871,80.967371,8.601812,52.59890,8.319781,5.55468,27.96,71.0,1012.0,3.90
1,gov,31390912357075263208060500522,08/04/2025 14:00,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,10.7823,106.7528,95,95.340935,59.552588,NaN,8.45665,NaN,16.50532,35.01,46.0,1009.0,1.54
2,gov,31390932574706768021562473002,08/04/2025 14:00,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,10.5391,106.4045,89,88.681186,52.501889,29.512184,NaN,NaN,1.95932,29.45,53.0,1009.0,5.84
3,gov,31390903576425084107499649578,08/04/2025 14:00,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),21.0052,105.8418,151,151.418019,72.311020,NaN,12.98290,9.472219,2.94200,28.00,70.0,1012.0,3.85
4,gov,31390908889087377344742439468,08/04/2025 14:00,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...,21.0031,105.7947,107,106.968661,71.106262,12.374913,4.44125,9.902344,2.29400,28.01,69.0,1012.0,3.68


In [7]:
unique_stations = df_aqi['Name'].unique()
station_df = df_aqi.groupby('Name')[['ID','Latitude', 'Longitude']].first().reset_index()
station_df

,Name,ID,Latitude,Longitude
0,"Bình Dương: số 593 Đại lộ Bình Dương, P. Hiệp ...",31388839920718814259329251882,10.99230,106.65770
1,Bắc Giang: Khu liên cơ quan tỉnh Bắc Giang - P...,31387251434693138681789561386,21.30150,106.22603
2,HCM: Khu Liên cơ quan Bộ Tài Nguyên và Môi Trư...,31390916083317566102523755051,10.78230,106.68340
3,HCM: Đ. Lê Hữu Kiều - P. Bình Trưng Tây - Quận...,31390912357075263208060500522,10.78230,106.75280
4,Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...,31388883344354363840031242796,20.53600,105.91650
5,Hà Nội: 556 Nguyễn Văn Cừ (KK),28560877461938780203765592307,21.04910,105.88310
6,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...,31390908889087377344742439468,21.00310,105.79470
7,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),31390903576425084107499649578,21.00520,105.84180
8,Long An: UBND Tp Tân An - 76 Hùng Vương - P.2 ...,31390932574706768021562473002,10.53910,106.40450
9,Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK),28505268571336961948594948504,21.33847,105.36330


In [8]:
OUTPUT_CSV = "/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data_2_0.csv"


In [9]:


API_KEY = "b3c6503ac31cb5e72765e98cd0d7795e"  # Your API Key

# Station IDs (Same as before)
AQI_IDS = ["31390903576425084107499649578", "28560877461938780203765592307", "31390908889087377344742439468", 
           "31390912357075263208060500522", "31390916083317566102523755051", "31388851800421997746903202346",
           "29195707587706641566224751462", "28505268571336961948594948504", "31387251434693138681789561386",
           "31388883344354363840031242796", "31390932574706768021562473002", "31388839920718814259329251882",
           "29213751141295132066317063859"]
WEATHER_IDS = ["1581130", "1561096", "1581364", "1566083", "1566083", "1594018",
               "1566319", "1562820", "1591527", "1570449", "1567069", "1565022", "1568839"]  


In [10]:

# Create a dictionary to map ID -> Weather ID
ID_MAPPING = dict(zip(AQI_IDS, WEATHER_IDS))

# --- HELPER FUNCTIONS ---

def convert_to_unix(timestamp_val):
    """
    Converts various timestamp formats to Unix timestamp (seconds).
    """
    try:
        if isinstance(timestamp_val, (int, float)):
            return int(timestamp_val)
        
        if isinstance(timestamp_val, str) and timestamp_val.replace('.', '', 1).isdigit():
            return int(float(timestamp_val))

        formats = [
            "%d/%m/%Y %H:%M",       
            "%Y-%m-%d %H:%M:%S",    
            "%d/%m/%Y %H:%M:%S"     
        ]
        
        for fmt in formats:
            try:
                dt = datetime.strptime(timestamp_val, fmt)
                return int(dt.timestamp())
            except ValueError:
                continue
        return None
    except Exception as e:
        print(f"Error parsing timestamp {timestamp_val}: {e}")
        return None

def fetch_history_by_id(city_id, start_time):
    """
    Fetches historical weather data using City ID.
    """
    url = "https://history.openweathermap.org/data/2.5/history/city"
    
    # We set end to start + 1 hour to define a valid window, 
    # but strictly requesting cnt=1 to get the closest/first record.
    params = {
        "id": city_id,
        "type": "hour",
        "start": start_time,
        "end": start_time, # Looking for exact or closest hour match
        "cnt": 1,
        "units": "metric",
        "appid": API_KEY
    }

    try:
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            data = response.json()
            if "list" in data and len(data["list"]) > 0:
                item = data["list"][0]
                return {
                    # New Columns
                    "Wind Degree": item.get("wind", {}).get("deg"),
                    "Wind Gust": item.get("wind", {}).get("gust"),
                    "Cloud Coverage": item.get("clouds", {}).get("all"),
                    "Visibility": item.get("visibility"),
                    "Weather Condition": item["weather"][0]["main"] if item.get("weather") else None,
                    "Weather Description": item["weather"][0]["description"] if item.get("weather") else None,
                    
                    # Standard Columns
                    "Temperature": item.get("main", {}).get("temp"),
                    "Pressure": item.get("main", {}).get("pressure"),
                    "Humidity": item.get("main", {}).get("humidity"),
                    "Wind Speed": item.get("wind", {}).get("speed")
                }
            else:
                print(f"No data found for ID: {city_id}, Time: {start_time}")
        else:
            print(f"API Error {response.status_code}: {response.text}")
            
    except Exception as e:
        print(f"Request failed: {e}")
        
    return {}

def main():
    if not os.path.exists(INPUT_CSV):
        print(f"File {INPUT_CSV} not found.")
        return

    print(f"Reading {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)

    # Initialize New Columns
    new_cols = ["Wind Degree", "Wind Gust", "Cloud Coverage", "Visibility", "Weather Condition", "Weather Description"]
    for col in new_cols:
        if col not in df.columns:
            df[col] = None
    
    # Initialize Standard Columns
    standard_cols = ["Temperature", "Pressure", "Humidity", "Wind Speed"]
    for col in standard_cols:
        if col not in df.columns:
            df[col] = None

    if "ID" not in df.columns:
        print("ERROR: 'ID' column not found in CSV. Cannot map to Weather IDs.")
        return

    print(f"Processing {len(df)} rows...")

    for index, row in df.iterrows():
        # Check if we already have the primary new data (Wind Degree)
        if pd.notna(row["Wind Degree"]):
            continue

        # Get ID and Map to Weather ID
        source_id = str(row.get("ID"))
        weather_id = ID_MAPPING.get(source_id)
        
        if not weather_id:
            print(f"Row {index}: ID '{source_id}' not found in mapping. Skipping.")
            continue

        raw_time = row.get("Timestamp")
        unix_time = convert_to_unix(raw_time)
        if not unix_time:
            continue

        print(f"Fetching data for Row {index} (ID: {weather_id} @ {raw_time})...")
        
        weather_data = fetch_history_by_id(weather_id, unix_time)
        
        if weather_data:
            # Set new columns
            df.at[index, "Wind Degree"] = weather_data.get("Wind Degree")
            df.at[index, "Wind Gust"] = weather_data.get("Wind Gust")
            df.at[index, "Cloud Coverage"] = weather_data.get("Cloud Coverage")
            df.at[index, "Visibility"] = weather_data.get("Visibility")
            df.at[index, "Weather Condition"] = weather_data.get("Weather Condition")
            df.at[index, "Weather Description"] = weather_data.get("Weather Description")
            
            # Fill standard columns if missing
            if pd.isna(row["Temperature"]):
                df.at[index, "Temperature"] = weather_data.get("Temperature")
            if pd.isna(row["Pressure"]):
                df.at[index, "Pressure"] = weather_data.get("Pressure")
            if pd.isna(row["Humidity"]):
                df.at[index, "Humidity"] = weather_data.get("Humidity")
            if pd.isna(row["Wind Speed"]):
                df.at[index, "Wind Speed"] = weather_data.get("Wind Speed")

        time.sleep(1.5) 
        
        if index % 10 == 0:
            df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"Done! Updated data saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Reading /home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv...
Processing 84862 rows...
Fetching data for Row 0 (ID: 1561096 @ 08/04/2025 14:00)...
API Error 401: {"cod":401, "message": "Invalid API key. Please see http://openweathermap.org/faq#error401 for more info."}
Fetching data for Row 1 (ID: 1566083 @ 08/04/2025 14:00)...
API Error 401: {"cod":401, "message": "Invalid API key. Please see http://openweathermap.org/faq#error401 for more info."}
Fetching data for Row 2 (ID: 1567069 @ 08/04/2025 14:00)...
API Error 401: {"cod":401, "message": "Invalid API key. Please see http://openweathermap.org/faq#error401 for more info."}
Fetching data for Row 3 (ID: 1581130 @ 08/04/2025 14:00)...
API Error 401: {"cod":401, "message": "Invalid API key. Please see http://openweathermap.org/faq#error401 for more info."}
Fetching data for Row 4 (ID: 1581364 @ 08/04/2025 14:00)...
API Error 401: {"cod":401, "message": "Invalid API key. Please see http://openweathermap.org/faq#

KeyboardInterrupt: 

In [ ]:
df_aqi_2 = pd.read_csv(OUTPUT_CSV)
df_aqi_2.head()